# Batch Dubbing (200+ videos) — Colab + Google Drive

Dub a whole folder of videos with ONE shared reference voice. Reads videos
from Google Drive and writes the dubbed videos back to Drive. **Resumable:**
if Colab disconnects, just run it again — it skips the ones already done.

### One-time setup in your Google Drive
Create this folder structure in **My Drive**:
```
VoiceDub/
  input/        <- put all your videos here (200+)
  output/       <- dubbed videos appear here
  reference.wav <- your ONE voice sample (same voice for all)
```
Then: `Runtime -> Change runtime type -> GPU -> Save`, and Run all.

In [ ]:
!nvidia-smi -L || echo 'Enable GPU: Runtime -> Change runtime type -> GPU'

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## 2. Get the project (auto-updates)

In [ ]:
import os
USER='PrashantRMarathe'; REPO='voice-replacer'
if os.path.isdir(REPO):
    !cd {REPO} && git pull --quiet
else:
    !git clone https://github.com/{USER}/{REPO}.git
%cd {REPO}
print('Now in:', os.getcwd())

## 3. Install dependencies (~5 min, first run only)

In [ ]:
!pip install -q coqui-tts 'transformers<5' ffmpeg-python deep-translator num2words soundfile openai-whisper demucs
!pip install -q f5-tts 2>/dev/null || echo 'NOTE: f5-tts not installed; will use XTTS.'
print('Dependencies installed.')

## 4. Settings + folders

Adjust the Drive paths if your folder is named differently.

In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'
import config
config.WHISPER_MODEL = 'large-v3'
config.WHISPER_BACKEND = 'openai'
config.TTS_BACKEND = 'f5'            # softest for English (falls back to xtts)
config.ENABLE_VOICE_ENHANCE = False  # keep voice soft
config.ENABLE_AUDIO_CLEANUP = False
config.PRESERVE_BACKGROUND = True    # keep original music/SFX
config.GENERATE_SUBTITLES = True

# ---- Your Drive folders ----
BASE = '/content/drive/MyDrive/VoiceDub'
INPUT_DIR = f'{BASE}/input'
OUTPUT_DIR = f'{BASE}/output'
REFERENCE = f'{BASE}/reference.wav'   # your one shared voice sample

SOURCE = 'en'    # spoken language (your videos are English)
TARGET = None    # None = keep English, just replace the voice

assert os.path.isdir(INPUT_DIR), f'Missing {INPUT_DIR} - create it in Drive and add videos.'
assert os.path.isfile(REFERENCE), f'Missing {REFERENCE} - put your voice sample there.'
import batch
print('Videos found:', len(batch.list_videos(INPUT_DIR)))

## 5. Run the batch (resumable)

Processes every video, writing `<name>_dubbed.mp4` to the Drive output folder.
If Colab disconnects, just re-run this cell — it skips finished ones.

In [ ]:
import batch
summary = batch.process_folder(
    INPUT_DIR, OUTPUT_DIR, REFERENCE,
    source_lang=SOURCE, target_lang=TARGET, cb=print,
)
print('\nSUMMARY:', summary)